# PLECS Simulation Runner
### Parameter Sweep & Monte Carlo — General Purpose Interface

**Workflow:** Configure → Connect → Run → Inspect signals → Plot → Save → Close

---
### How signals flow from PLECS to Python

Add an **Outport** block on the **top-level** schematic of your model  
(Component → Signal Sources → Outport). Wire all signals into it via a **Mux** block.

| `signal_index` | Signal |
|:-:|---|
| 0 | first signal into the Outport (top of Mux) |
| 1 | second signal |
| … | … |

Run **Inspect signals** after the first simulation to see the exact mapping.

---
### BASE\_VARS modes

| Mode | `BASE_VARS` | What PLECS receives |
|---|---|---|
| **A – Python controls all** | populated dict | every listed variable + swept values |
| **B – PLECS controls all** | `{}` | only the swept / MC values |

---
### Plot performance

Plots use **WebGL rendering** (GPU) and **min-max vectorised downsampling**  
(fully numpy, no Python loop — < 1 ms per trace even on 1 M-point signals).

The `MAX_POINTS` budget is **shared across all traces** in a figure,  
so browser load stays constant regardless of run count.

> Requires `plecs_sim_library.py` in the same folder as this notebook  
> and PLECS Standalone running with the XML-RPC interface enabled (port 1080).

In [1]:
from src.plecs_sim_library import (
    connect_plecs, load_model, close_model,
    inspect_signals,
    run_sweep, run_montecarlo,
    plot_signals, plot_montecarlo_histogram,
    save_results_csv, save_montecarlo_stats_csv,
)
import numpy as np

---
##  Configuration
**This is the only cell you normally need to edit.**

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
#  1. PLECS XML-RPC connection
# ─────────────────────────────────────────────────────────────────────────────
PLECS_HOST = "http://localhost:1080/RPC2"   # change port if needed

# ─────────────────────────────────────────────────────────────────────────────
#  2. Model location
#  #     MODEL_FOLDER : absolute path to the folder containing the .plecs file
#  #     MODEL_NAME   : filename WITHOUT the .plecs extension
# ─────────────────────────────────────────────────────────────────────────────
MODEL_FOLDER = r"C:\Users\eliott.sefarang\OneDrive - HESSO\Assistanat\PyPLECS\SimulationPyPLECS"
MODEL_NAME   = "20260220_ETPS_DoublePhaseBuck60kHz"      # <-- edit this

# ─────────────────────────────────────────────────────────────────────────────
#  3. BASE_VARS
#  #  Mode A -- Python controls everything: populate the dict below.
#  #  Mode B -- PLECS controls everything: set BASE_VARS = {}
# ─────────────────────────────────────────────────────────────────────────────
# Mode A example (uncomment and fill in):
# BASE_VARS = {
#     'R': 0.5, 'L': 3e-3, 'fc': 20e3,
#     'Ki': 0.5/(4*0.5*1.5*(1/20e3)*1*(1/0.5)),
#     'Ti': 3e-3/0.5, 'Istep': 10,
#     't_switching': 0.03, 't_stepRef': 0.23, 't_end': 0.73,
# }
BASE_VARS = {}   # <-- populated dict = Mode A | empty = Mode B

# ─────────────────────────────────────────────────────────────────────────────
#  4. Execution mode
#  #     'simulate' -> run time-domain simulation
#  #     'analyze'  -> run steady-state or other analysis
# ─────────────────────────────────────────────────────────────────────────────
EXEC_MODE = 'simulate'   # <-- 'simulate' or 'analyze'

# ─────────────────────────────────────────────────────────────────────────────
#  5. Analysis name  (used when EXEC_MODE = 'analyze')
# ─────────────────────────────────────────────────────────────────────────────
ANALYSIS_NAME = 'Steady-State Analysis'   # <-- name of the analysis in PLECS

# ─────────────────────────────────────────────────────────────────────────────
#  6. Simulation mode
#  #     'sweep'      -> vary parameters over a fixed list of values
#  #     'montecarlo' -> randomly vary parameters (statistical analysis)
# ─────────────────────────────────────────────────────────────────────────────
SIM_MODE = 'sweep'   # <-- 'sweep' or 'montecarlo'

# ─────────────────────────────────────────────────────────────────────────────
#  5a. Sweep parameters  (used when SIM_MODE = 'sweep')
#  #
#  #  Format:  { 'PLECS_var_name' : [val0, val1, ...], ... }
#  #  Single parameter -> 1-D sweep.
#  #  Multiple params  -> stepped together (zip); all lists same length.
# ─────────────────────────────────────────────────────────────────────────────

SWEEP_PARAMS = {
    'Therm_mod' : ['file:IMZA120R012M2H-SKG', 'file:IMZA120R017M2H-SKG', 'file:IMZA120R022M2H-SKG', 'file:IMZA120R026M2H-SKG'],
    # 'L' : [2e-3, 3e-3, 4e-3, 5e-3],
}
SWEEP_LABEL_PARAM = 'Therm_mod'   # parameter shown in the legend

# ─────────────────────────────────────────────────────────────────────────────
#  5b. Monte Carlo parameters  (used when SIM_MODE = 'montecarlo')
#  #
#  #  Format:  { 'PLECS_var_name' : (nominal, rel_tolerance, distribution) }
#  #  distribution: 'uniform' (flat +/-tol) or 'normal' (sigma = nominal*tol)
# ─────────────────────────────────────────────────────────────────────────────
MC_PARAMS = {
    'R' : (0.5,  0.10, 'uniform'),   # +/-10 % flat
    'L' : (3e-3, 0.05, 'normal'),    # 5 % sigma
}
N_MC_RUNS = 20
MC_SEED   = 42    # None = random each time

# ─────────────────────────────────────────────────────────────────────────────
#  6. Signal names
#  #
#  #  Name every signal that comes out of the PLECS Outport, in wiring order
#  #  (top to bottom in the Mux block).
#  #  Use inspect_signals(results) after the first run if you are unsure.
# ─────────────────────────────────────────────────────────────────────────────
SIGNAL_NAMES = [
    'Junction temperature Ph1 HS',   # signal_index = 0
    'Junction temperature Ph1 LS',  # signal_index = 1
    # 'Iq_meas_A',
]

# ─────────────────────────────────────────────────────────────────────────────
#  7. Plot configuration
#  #
#  #  One dict per figure:
#  #
#  #  signal_index  int or list[int]
#  #      0 = first signal into PLECS Outport, 1 = second, etc.
#  #      Several signals on one figure: signal_index=[0, 1]
#  #
#  #  signal_name   str or list[str]  -- display name(s), must match signal_index
#  #
#  #  time_window   (t_start, t_end) in seconds, or None (full simulation).
#  #      TIP: A narrow window is faster because downsampling only processes
#  #           the visible portion of the data.
#  #
#  #  title / xlabel / ylabel  strings
#  #
#  #  show_legend   bool (default True)
#  #      Set False when many runs make the legend unreadable.
# ─────────────────────────────────────────────────────────────────────────────
PLOT_CONFIG = [
    {
        'signal_index' : [0, 1],
        'signal_name'  : 'Junction temperature Ph HS',
        'time_window'  : None,#(0.0, 0.9),   
        'title'        : 'Junction temperature',
        'xlabel'       : 'Time (s)',
        'ylabel'       : 'Temperature (°C)',
        'show_legend'  : True,
    },
    {
        'signal_index' : 1,
        'signal_name'  : 'Junction temperature Ph LS',
        'time_window'  : None,     # <- full simulation time
        'title'        : 'Full run',
        'ylabel'       : 'Temperature (°C)',
        'show_legend'  : False,    # <- hide legend (many runs)
    },
]

# ─────────────────────────────────────────────────────────────────────────────
#  8. Plot performance: MAX_POINTS (total point budget per figure)
#  #
#  #  The budget is shared across ALL traces in a figure.
#  #  Each trace gets  MAX_POINTS // n_traces  rendered points.
#  #  Min-max vectorised downsampling runs < 1 ms/trace even on 1M-pt signals.
#  #  The CSV export always saves the full raw data regardless of this setting.
#  #
#  #  Recommended values
#  #  ------------------
#  #  50 000   default -- good for sweep (few traces, high detail)
#  #  20 000   for 20-100 MC runs
#  #  10 000   for 100-500 MC runs
#  #  None     no downsampling (only for already-small datasets)
# ─────────────────────────────────────────────────────────────────────────────
MAX_POINTS = 50_000   # <-- adjust based on run count (see table above)

# ─────────────────────────────────────────────────────────────────────────────
#  9. Monte Carlo histogram  (used when SIM_MODE = 'montecarlo')
# ─────────────────────────────────────────────────────────────────────────────
MC_HISTOGRAM = {
    'signal_index' : 1,
    't_eval'       : 0.232,
    'signal_name'  : 'Id meas (A)',
    'title'        : 'MC distribution of Id at t = 0.232 s',
    'xlabel'       : 'Current (A)',
    'n_bins'       : 15,
    'show_legend'  : True,
}

# ─────────────────────────────────────────────────────────────────────────────
#  10. CSV export
#  #  SAVE_CSV    True/False
#  #  CSV_FOLDER  target folder (created if it does not exist)
#  #  MC_STATS_TIMES  time instants sampled for the MC compact stats CSV
# ─────────────────────────────────────────────────────────────────────────────
SAVE_CSV       = False
CSV_FOLDER     = r"C:\Users\you\sim_data"   # <-- edit this
MC_STATS_TIMES = [0.230, 0.232, 0.234]

# Summary
_mode = 'A — Python controls all vars' if BASE_VARS else 'B — PLECS controls all vars'
print('Configuration loaded.')
print(f'  BASE_VARS mode  : {_mode}')
print(f'  Simulation mode : {SIM_MODE}')
print(f'  MAX_POINTS      : {MAX_POINTS}')
print(f'  CSV export      : {SAVE_CSV}')
if SAVE_CSV:
    print(f'  CSV folder      : {CSV_FOLDER}')
if BASE_VARS:
    print(f'  BASE_VARS keys  : {list(BASE_VARS.keys())}')


Configuration loaded.
  BASE_VARS mode  : B — PLECS controls all vars
  Simulation mode : sweep
  MAX_POINTS      : 50000
  CSV export      : False


---
##  Step 1 — Connect to PLECS and Load the Model

In [3]:
server = connect_plecs(PLECS_HOST)
load_model(server, MODEL_FOLDER, MODEL_NAME)

[PLECS] Connected to http://localhost:1080/RPC2


KeyboardInterrupt: 

---
##  Step 2 — Run Simulations

In [ ]:
if SIM_MODE == 'sweep':
    results = run_sweep(server, MODEL_NAME, BASE_VARS, SWEEP_PARAMS,
                        mode=EXEC_MODE, analysis_name=ANALYSIS_NAME if EXEC_MODE == 'analyze' else None)

elif SIM_MODE == 'montecarlo':
    results = run_montecarlo(server, MODEL_NAME, BASE_VARS, MC_PARAMS,
                             n_runs=N_MC_RUNS, seed=MC_SEED,
                             mode=EXEC_MODE, analysis_name=ANALYSIS_NAME if EXEC_MODE == 'analyze' else None)
else:
    raise ValueError(f"Unknown SIM_MODE '{SIM_MODE}'.")

[Sweep] 4 step(s) | swept: ['Therm_mod'] | mode: PLECS vars only (base_vars empty)
  step   1/4  ->  {'Therm_mod': 'file:IMZA120R012M2H-SKG'}
  step   2/4  ->  {'Therm_mod': 'file:IMZA120R017M2H-SKG'}
  step   3/4  ->  {'Therm_mod': 'file:IMZA120R022M2H-SKG'}
  step   4/4  ->  {'Therm_mod': 'file:IMZA120R026M2H-SKG'}
[Sweep] Done.


---
## Step 3 — Inspect Signals *(run once to map signal indices)*
Prints index, min, max, and mean for every signal from the PLECS Outport.  
Use these indices in `signal_index` inside `PLOT_CONFIG`.

In [ ]:
inspect_signals(results)

Signals available in results (first run):
  Total signals  : 2  -> use signal_index = 0 to 1
  Total samples  : 3244740
  Time range     : 0 s  to  0.9 s

  index                min             max            mean
  -------    -------------   -------------   -------------
  0                 108.01          114.31          110.68
  1                 100.37          112.71          104.82


---
## Step 4 — Plot Results
Charts are interactive: scroll to zoom, drag to pan, hover for exact values,  
click legend entries to hide/show individual runs.

A downsampling summary is printed below each figure showing the raw vs  
rendered point counts and the total build time.

In [ ]:
plot_signals(
    results,
    PLOT_CONFIG,
    sim_mode    = SIM_MODE,
    label_param = SWEEP_LABEL_PARAM if SIM_MODE == 'sweep' else None,
    max_points  = MAX_POINTS,
)


[Plot] 'Junction temperature'
       8 trace(s) | 12,978,960 raw pts -> 25,004 rendered (519x reduction) | build 0.07s | render 0.08s


[Plot] 'Full run'
       4 trace(s) | 12,978,960 raw pts -> 50,004 rendered (260x reduction) | build 0.08s | render 0.08s


In [ ]:
if SIM_MODE == 'montecarlo':
    plot_montecarlo_histogram(
        results,
        signal_index = MC_HISTOGRAM['signal_index'],
        t_eval       = MC_HISTOGRAM['t_eval'],
        signal_name  = MC_HISTOGRAM['signal_name'],
        title        = MC_HISTOGRAM['title'],
        xlabel       = MC_HISTOGRAM['xlabel'],
        n_bins       = MC_HISTOGRAM['n_bins'],
        show_legend  = MC_HISTOGRAM['show_legend'],
    )
else:
    print('Histogram skipped (SIM_MODE is not montecarlo).')


Histogram skipped (SIM_MODE is not montecarlo).


---
## Step 5 — Save to CSV *(optional)*
- **All modes** → full time-series CSV (raw data, not downsampled)
- **Monte Carlo** → also writes a compact stats table sampled at `MC_STATS_TIMES`

In [ ]:
if SAVE_CSV:
    save_results_csv(results, signal_names=SIGNAL_NAMES, save_folder=CSV_FOLDER)
    if SIM_MODE == 'montecarlo':
        save_montecarlo_stats_csv(results, signal_names=SIGNAL_NAMES,
                                  t_eval_list=MC_STATS_TIMES, save_folder=CSV_FOLDER)
else:
    print('CSV export skipped (SAVE_CSV = False).')


CSV export skipped (SAVE_CSV = False).


---
## Step 6 — Close the PLECS Model

In [ ]:
close_model(server, MODEL_NAME)

[PLECS] Model '20260220_ETPS_DoublePhaseBuck60kHz' closed.
